In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('/kaggle/input/datasets/organizations/uciml/pima-indians-diabetes-database/diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
df.corr()['Outcome']

Pregnancies                 0.221898
Glucose                     0.466581
BloodPressure               0.065068
SkinThickness               0.074752
Insulin                     0.130548
BMI                         0.292695
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

In [5]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [6]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [7]:
X = scaler.fit_transform(X)

In [8]:
X.shape

(768, 8)

In [9]:
from sklearn.model_selection import train_test_split

X_train,X_test ,y_train,y_test = train_test_split(X,y, test_size=0.2,random_state=1)

In [10]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Dropout

In [11]:
model = Sequential()

model.add(Dense(32,activation='relu',input_dim=8))
model.add(Dense(1,activation='sigmoid'))

model.compile(loss='binary_crossentropy',optimizer='Adam',metrics=['accuracy'])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-07-26 12:00:47.182669: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [12]:
model.fit(X_train,y_train,batch_size=32,epochs=10,validation_data=(X_test,y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6987 - loss: 0.5936 - val_accuracy: 0.7792 - val_loss: 0.5562
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7117 - loss: 0.5603 - val_accuracy: 0.7727 - val_loss: 0.5301
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7182 - loss: 0.5373 - val_accuracy: 0.7987 - val_loss: 0.5104
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7362 - loss: 0.5219 - val_accuracy: 0.7857 - val_loss: 0.4973
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7394 - loss: 0.5094 - val_accuracy: 0.8117 - val_loss: 0.4864
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7427 - loss: 0.4992 - val_accuracy: 0.8182 - val_loss: 0.4759
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7492 - loss: 0.4907 - val_accuracy: 0.7922 - val_loss: 0.4694
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7622 - loss: 0.4839 - val_accuracy: 0.7987 - val_loss

# 1. How to select the appropriate Optimizer
# 2. No of nodes in the layer
# 3. Num of layers
# 4. All in all one model

In [13]:
import keras_tuner as tk

In [14]:
def build_model(hp):

    model = Sequential()
    model.add(Dense(32,activation='relu',input_dim=8))
    model.add(Dense(1,activation='sigmoid'))

    optimizer = hp.Choice('optimizer',values = ['adam','sgd','adagrad','adadelta,rmsprop'])
    model.compile(loss='binary_crossentropy',optimizer=optimizer,metrics =['accuracy'])

    return model
    

In [15]:
tuner = tk.RandomSearch(build_model,objective='val_accuracy',max_trials=5)


In [16]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 4 Complete [00h 00m 02s]
val_accuracy: 0.6363636255264282

Best val_accuracy So Far: 0.7402597665786743
Total elapsed time: 00h 00m 08s


# all in one

In [17]:
def build_model(hp):
    model =Sequential()
    counter=0
    for i in range(hp.Int('num_layers',min_value=1,max_value=10)):
        if counter==0:
            model.add(Dense(hp.Int('unit'+str(i), min_value=8,max_value=128,step=8),activation=hp.Choice('activation'+str(i),values=['relu','tanh','sigmoid']),input_dim=8))
            model.add(Dropout(hp.Choice('Dropout'+str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))

        else:
            model.add(Dense(hp.Int('unit'+str(i), min_value=8,max_value=128,step=8),activation=hp.Choice('activation'+str(i),values=['relu','tanh','sigmoid'])))
            model.add(Dropout(hp.Choice('Dropout'+str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))

        counter+=1
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer=hp.Choice('optimizer',values=['rmsprop','adam','sgd','nadam','adadelta']),
                 loss='binary_crossentropy',
                 metrics=['accuracy'])

    return model

In [18]:
tuner = tk.RandomSearch(build_model,
                       objective='val_accuracy',
                       max_trials=3,
                       directory='mydir',
                       project_name='final')

In [19]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 03s]
val_accuracy: 0.6428571343421936

Best val_accuracy So Far: 0.6428571343421936
Total elapsed time: 00h 00m 13s


In [21]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 7,
 'unit0': 64,
 'activation0': 'tanh',
 'Dropout0': 0.8,
 'optimizer': 'nadam',
 'unit1': 8,
 'activation1': 'relu',
 'Dropout1': 0.1,
 'unit2': 8,
 'activation2': 'relu',
 'Dropout2': 0.1,
 'unit3': 8,
 'activation3': 'relu',
 'Dropout3': 0.1,
 'unit4': 8,
 'activation4': 'relu',
 'Dropout4': 0.1,
 'unit5': 8,
 'activation5': 'relu',
 'Dropout5': 0.1,
 'unit6': 8,
 'activation6': 'relu',
 'Dropout6': 0.1}

In [22]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'nadam', because it has 2 variables whereas the saved optimizer has 35 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [23]:
model.fit(X_train,y_train,epochs=100,initial_epoch=5,validation_data=(X_test,y_test))

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.6547 - loss: 0.6724 - val_accuracy: 0.6429 - val_loss: 0.6662
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6629 - loss: 0.6596 - val_accuracy: 0.6429 - val_loss: 0.6503
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6515 - loss: 0.6441 - val_accuracy: 0.6429 - val_loss: 0.6323
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6661 - loss: 0.6269 - val_accuracy: 0.6429 - val_loss: 0.6118
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6547 - loss: 0.6207 - val_accuracy: 0.6429 - val_loss: 0.5842
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6694 - loss: 0.6083 - val_accuracy: 0.6753 - val_loss: 0.5596
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6661 - loss: 0.5973 - val_accuracy: 0.7208 - val_loss: 0.5409
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6775 - loss: 0.5848 - val_accuracy: 0.727